# Module 05 · Limit Hold'em tools

This notebook exercises the hand evaluator and range-equity helpers from the repository. Exact enumeration is kept to a flop or later; preflop enumeration is intentionally not attempted.

In [ ]:
from pathlib import Path
import sys
for parent in (Path.cwd(), *Path.cwd().parents):
    if (parent / 'pokerbot').is_dir():
        sys.path.insert(0, str(parent))
        break
else:
    raise RuntimeError('Run this notebook inside the pokerbots repository')

## Rank five and seven cards

A card is encoded as `suit * 13 + rank - 2`, with ranks 2 through Ace and suits 0 through 3. Returned tuples compare lexicographically; the first item is the category.

In [ ]:
from pokerbot.eval.hand_eval import evaluate_five, evaluate_seven, compare_hands

def card(rank, suit):
    return suit * 13 + rank - 2

royal_flush = [card(rank, 0) for rank in range(10, 15)]
wheel = [card(14, 0), card(2, 1), card(3, 2), card(4, 3), card(5, 0)]
print('royal flush:', evaluate_five(royal_flush))
print('wheel straight:', evaluate_five(wheel))
seven_cards = royal_flush + [card(9, 1), card(8, 2)]
print('best five of seven:', evaluate_seven(seven_cards))
assert evaluate_five(royal_flush) == (8, 14)
assert evaluate_five(wheel) == (4, 5)
assert evaluate_seven(seven_cards) == (8, 14)

## Exact river equity and blockers

Each listed villain combo is equally likely. Combos that overlap the hero or board are removed before evaluation.

In [ ]:
from pokerbot.eval.equity import enumerate_equity, estimate_equity

hero = [card(14, 0), card(14, 1)]
board = [card(2, 0), card(3, 1), card(4, 2), card(8, 3), card(9, 0)]
villain_range = [[card(13, 2), card(13, 3)], [card(14, 2), card(14, 3)]]
exact = enumerate_equity(hero, board, villain_range)
print(exact)
assert exact.wins == 1 and exact.ties == 1
assert exact.equity == 0.75

## Monte Carlo on a flop

There are many turn-and-river runouts, so sample them and report the standard error. Change the seed and sample count to see how estimates vary.

In [ ]:
flop = [card(2, 0), card(7, 1), card(11, 2)]
villain_range = [[card(13, 2), card(13, 3)], [card(12, 0), card(12, 1)]]
exact_flop = enumerate_equity(hero, flop, villain_range)
sampled = estimate_equity(hero, flop, villain_range, samples=3000, seed=19)
print(f'exact equity: {exact_flop.equity:.3f}')
print(f'sampled equity: {sampled.equity:.3f} ± {sampled.standard_error:.3f} (standard error)')
assert abs(sampled.equity - exact_flop.equity) < 0.08

## Starting-hand classes and next steps

A starting hand class is either a pair, suited non-pair, or offsuit non-pair. This gives 13 + 78 + 78 = 169 strategic classes, while the physical deck has many more individual combos.

The full four-street Limit Hold'em engine and abstract strategy solver are the next implementation steps. Try adding weighted ranges, a hand-strength histogram, and a fixed-limit action state machine.

In [ ]:
pairs = 13
nonpair_rank_combinations = 13 * 12 // 2
classes = pairs + nonpair_rank_combinations + nonpair_rank_combinations
print('starting hand classes:', classes)
assert classes == 169